In [17]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from matplotlib.colors import LinearSegmentedColormap, Normalize
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from mpl_toolkits.mplot3d.art3d import Line3DCollection

In [18]:

# cols [trials,threads,pi,error,time]
data = open('out.txt','r')
D = []
for line in data:
    L = line.split(',')
    for i in range(3):
        L[i] = int(L[i].split(' ')[-1])
    L[3] = float(L[3].split(' ')[-2])
    D.append(L)
labels = ['Matrix Size','BlockX','BlockY','time (ms)']
print(labels)
# for i in D:
#     print(i)
D = np.array(D)
# convert ms -> s
D[:, 3] = D[:, 3] / 1000.0
print(D)

['Matrix Size', 'BlockX', 'BlockY', 'time (ms)']
[[4.09600000e+03 3.20000000e+01 3.20000000e+01 4.56795100e-01]
 [4.09600000e+03 6.40000000e+01 1.60000000e+01 3.88900900e-01]
 [4.09600000e+03 1.60000000e+01 6.40000000e+01 4.39737300e-01]
 [4.09600000e+03 1.28000000e+02 8.00000000e+00 3.77023500e-01]
 [4.09600000e+03 8.00000000e+00 1.28000000e+02 4.30089200e-01]
 [4.09600000e+03 2.56000000e+02 4.00000000e+00 3.70105300e-01]
 [4.09600000e+03 4.00000000e+00 2.56000000e+02 5.11809500e-01]
 [4.09600000e+03 3.20000000e+01 1.60000000e+01 4.18085900e-01]
 [4.09600000e+03 1.60000000e+01 3.20000000e+01 4.62332900e-01]
 [4.09600000e+03 6.40000000e+01 8.00000000e+00 3.91843800e-01]
 [4.09600000e+03 8.00000000e+00 6.40000000e+01 4.54088700e-01]
 [4.09600000e+03 1.28000000e+02 4.00000000e+00 3.79069400e-01]
 [4.09600000e+03 4.00000000e+00 1.28000000e+02 5.26382100e-01]
 [4.09600000e+03 1.60000000e+01 1.60000000e+01 4.63947800e-01]
 [4.09600000e+03 3.20000000e+01 8.00000000e+00 4.13008900e-01]
 [4.09

In [23]:
# ---------------------------------------------------------------------------
# 2. Custom blue -> cyan -> green colormap
# ---------------------------------------------------------------------------
 
time_cmap = LinearSegmentedColormap.from_list(
    "time_bcg", ["#1f3fff", "#00e5ff", "#00c853"]
)
 
# ---------------------------------------------------------------------------
# 3. Helpers
# ---------------------------------------------------------------------------
 
matrix_sizes = sorted(set(D[:, 0].astype(int)))
all_threads = D[:, 1] * D[:, 2]
 
# discrete point-size scale: 4 / 8 / 12 / 16 pt (diameter), assigned by
# thread-count category so the same thread count always maps to the same
# size across all three figures.
SIZE_STEPS_PT = [4, 8, 12, 16]
unique_threads_global = sorted(set(int(x) for x in all_threads))
 
if len(unique_threads_global) <= len(SIZE_STEPS_PT):
    thread_to_pt = {th: SIZE_STEPS_PT[i] for i, th in enumerate(unique_threads_global)}
else:
    # bucket into quantile groups if more than 4 distinct thread counts appear
    edges_q = np.quantile(unique_threads_global, np.linspace(0, 1, len(SIZE_STEPS_PT) + 1))
    thread_to_pt = {}
    for th in unique_threads_global:
        bucket = min(np.searchsorted(edges_q, th, side="right") - 1, len(SIZE_STEPS_PT) - 1)
        thread_to_pt[th] = SIZE_STEPS_PT[bucket]
 
 
def thread_to_markersize_pt(threads):
    """threads: array-like of int thread counts -> array of marker diameters (pt)"""
    return np.array([thread_to_pt[int(x)] for x in np.atleast_1d(threads)])
 
 
output_files = []
 
for n in matrix_sizes:
    mask = D[:, 0].astype(int) == n
    sub = D[mask]
    bx = sub[:, 1]
    by = sub[:, 2]
    t = sub[:, 3]           # seconds
    threads = bx * by
 
    log_bx = np.log2(bx)
    log_by = np.log2(by)
 
    fig = plt.figure(figsize=(11, 9))
    ax = fig.add_subplot(111, projection="3d")
    # disable matplotlib's automatic depth-based z-ordering for 3D artists;
    # without this, text can render behind the surface/points depending on
    # viewing angle regardless of the zorder value passed to ax.text
    ax.computed_zorder = False
 
    t_norm = Normalize(vmin=t.min(), vmax=t.max())
 
    # --- triangulated surface (the "carpet") ---
    triang = mtri.Triangulation(log_bx, log_by)
    surf = ax.plot_trisurf(
        log_bx, log_by, t, triangles=triang.triangles,
        cmap=time_cmap, norm=t_norm,
        edgecolor="none", alpha=0.85, shade=True,
    )
    surf.set_zorder(1)
 
    # --- thin uniform wireframe on top, purely for the connected-carpet look ---
    edges = set()
    for tri_ in triang.triangles:
        for a, b in [(tri_[0], tri_[1]), (tri_[1], tri_[2]), (tri_[2], tri_[0])]:
            edges.add(tuple(sorted((a, b))))
 
    segments = [
        [(log_bx[a], log_by[a], t[a]), (log_bx[b], log_by[b], t[b])]
        for a, b in edges
    ]
    edge_collection = Line3DCollection(
        segments, colors="dimgray", linewidths=0.6, alpha=0.5
    )
    edge_collection.set_zorder(2)
    ax.add_collection3d(edge_collection)
 
    # vertex markers sized discretely by thread count (BlockX*BlockY)
    marker_pt = thread_to_markersize_pt(threads)
    ax.scatter(
        log_bx, log_by, t,
        c="black", s=marker_pt ** 2,
        edgecolors="white", linewidths=0.4,
        depthshade=False, zorder=3,
    )
 
    # --- label the 4 fastest and 4 slowest configs ---
    order = np.argsort(t)
    slowest_idx = [order[-4],order[-1]]
    fastest_idx = order[:4]
 
    def annotate(idx, color, prefix):
        x_range = log_bx.max() - log_bx.min()
        y_range = log_by.max() - log_by.min()
        z_range = t.max() - t.min()
        dx = 0.10 * max(x_range, 1e-6)
        dy = -0.10 * max(y_range, 1e-6)
        dz = 0.2 * max(z_range, 1e-6)
        for i in idx:
            lx, ly, lz = log_bx[i] + dx, log_by[i] + dy, t[i] + dz
            # leader line from the point to the offset label position
            ax.plot(
                [log_bx[i], lx], [log_by[i], ly], [t[i], lz],
                color=color, linewidth=0.8, linestyle="--", alpha=0.8, zorder=9,
            )
            ax.text(
                lx, ly, lz,
                f"{prefix} {int(bx[i])}x{int(by[i])}\n{t[i]:.3f}s",
                fontsize=7, color=color, weight="bold", zorder=10,
                bbox=dict(facecolor="white", edgecolor=color, alpha=0.9, pad=1.5),
            )
 
    annotate(fastest_idx, "#0057b8", "FAST")
    annotate(slowest_idx, "#c62828", "SLOW")
 
    # axis ticks: show actual block sizes, not log2 values
    tick_vals = sorted(set(list(bx.astype(int)) + list(by.astype(int))))
    tick_pos = np.log2(tick_vals)
    ax.set_xticks(tick_pos)
    ax.set_xticklabels([str(v) for v in tick_vals], rotation=45, ha="right", fontsize=8)
    ax.set_yticks(tick_pos)
    ax.set_yticklabels([str(v) for v in tick_vals], fontsize=8)
 
    ax.set_xlabel("BlockX", labelpad=12)
    ax.set_ylabel("BlockY", labelpad=12)
    ax.set_zlabel("Time (s)", labelpad=8)
    ax.set_title(f"Matrix Size = {n}", fontsize=14, fontweight="bold")
 
    mappable = plt.cm.ScalarMappable(cmap=time_cmap, norm=t_norm)
    mappable.set_array([])
    cbar = fig.colorbar(mappable, ax=ax, shrink=0.6, pad=0.1)
    cbar.set_label("Time (s)")
 
    # marker-size legend: every distinct thread-count category present here
    present_threads = sorted(set(int(x) for x in threads))
    legend_handles = [
        plt.scatter([], [], s=thread_to_pt[rt] ** 2, color="black",
                    edgecolors="white", linewidths=0.4, label=f"{rt} threads")
        for rt in present_threads
    ]
    ax.legend(handles=legend_handles, loc="upper left",
              title="Marker size =\nBlockX*BlockY", fontsize=8, title_fontsize=9,
              framealpha=0.9)
 
    ax.view_init(elev=25, azim=-60)
 
    outpath = f"carpet_n{n}.png"
    fig.savefig(outpath, dpi=160, bbox_inches="tight")
    output_files.append(outpath)
    plt.close(fig)
 
print("Saved:", output_files)

Saved: ['carpet_n4096.png', 'carpet_n8192.png', 'carpet_n16384.png']
